### 1. Initialization and read


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length, date_sub, lead


In [0]:
df = spark.table("catalogue_project1.bronze.erp_cust_az12")

In [0]:
%sql
-- df.limit(10).display()
select distinct gen from catalogue_project1.bronze.erp_cust_az12

### 2. Transformations
2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


2.2 Normalize customer ID

In [0]:
df = (
    df.withColumn(
        "cid",
        F.when(F.col("cid").startswith("NAS"),
            F.substring(col("cid"), 4, F.length(col("cid")))
        ).otherwise(col("cid"))
    )
)



2.3 Validate birth dates

In [0]:
df = df.withColumn(
    "bdate", 
    F.when(F.col("bdate") > F.current_date(), None)
    .otherwise(col("bdate")))


2.4 Standarize gender

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)


2.5 Rename columns

In [0]:
RENAME_CONFIG = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_CONFIG.items():
    df = df.withColumnRenamed(old_name, new_name)

* Intermediate step: check the changes applied correctly.

In [0]:
df.limit(10).display()

### 3. Write into the Silver delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.silver.erp_cust_az12")

* Check that everything went well

In [0]:
%sql
SELECT * FROM catalogue_project1.silver.erp_cust_az12 LIMIT 10
